# Optimization Project

In [1]:
# Setup: packages
import warnings
warnings.filterwarnings("ignore")

# General packages
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt

# Stock importing
import yfinance as yf

# Optimization
from gurobipy import *

ModuleNotFoundError: No module named 'gurobipy'

### Setup

In [ ]:
# All 5 stocks according to requirements
TICKERS = ["NVDA", "GOOG", "AVGO", "AAPL", "META"]
# Specify Start and End for proper returns
START = "2024-12-31" # IMPORTANT: Start at 12-31-24 so that we have returns starting on the 01-02-25. Off New Years
END = "2026-01-01" # IMPORTANT: End so it captures through 12-31

import time
time.sleep(2)  # Wait 2 seconds before requesting

# Pull adjusted closing prices
raw = yf.download(
    TICKERS,
    start=START,
    end=END,
    auto_adjust=True,
    progress=False,
    threads=False
)
# Extract closing prices
prices = raw["Close"][TICKERS]

# Calculate Daily Return Percentages
returns = prices.pct_change().dropna() * 100

### Recommendations

- Use the historical volatility and the mean of the percent daily return
- Optimize stock strategy to **minimize the risk for a daily return** of **at least 0.18%**
- Assume $\sum_{p_i} = 1$
- Assume no transaction costs to buy or sell stocks
- Provide a recommendation as to **what percent of this portfolio** to allocate in each stock

**SETUP:**

- **Decision variables:** 5 (what proportion should go into each investment)
- **Minimize daily risk:** $\sum_i \sum_j p_i \sigma_{i,j} p_j$
- **Constraints:**
    - Sum of proportions equals 1: $\sum_i p_i = 1$
    - Return of at least $0.18$: $\sum_i p_i r_i \geq 0.18$

In [ ]:
# TEMPLATE FROM OPTIMIZATION SLIDES

# SETUP
stocks = returns.columns # Stock names
num_stocks = len(stocks) # Number of stocks
# Already have expected returns
cov_mat = returns.cov() # Covariance matrix
m = Model('portfolio') # Creating empty model

# MODEL CREATION
vars = pd.Series(m.addVars(stocks, lb = 0), index = stocks) # Add a variable for each stock
portfolio_risk = cov_mat.dot(vars).dot(vars) # Define daily risk
m.setObjective(portfolio_risk, GRB.MINIMIZE) # Set objective function

# ADDING CONSTRAINTS
m.addConstr(vars.sum() == 1, 'Budget') # Sum of proportions must equal 1

mu = returns.mean() # Expected (mean) daily return per stock
m.addConstr(mu.dot(vars) >= 0.18, 'Daily Return') # Daily return of at least 0.18

# RUNNING MODEL AND OBTAINING SOLUTION
m.optimize()
print('Minimum Risk Portfolio:\n')
for v in vars:
    if v.x > 0:
        print('\t%s\t: %g' % (v.varname, round(v.x, 4)))

**SOLUTION:**
- **NVDA:** 0.0111 $\rightarrow$ **1.11%**
- **GOOG:** 0.7131 $\rightarrow$ **71.31%**
- **AVGO:** 0.0150 $\rightarrow$ **1.50%**
- **APPL:** 0.1573 $\rightarrow$ **15.73%**
- **META:** 0.1034 $\rightarrow$ **10.34%**

### Efficient Frontier

- Create the efficient frontier and provide a report-ready graph
- Provide a strategy on portfolio options for **different levels of risk**

In [ ]:
# TEMPLATE FROM OPTIMIZATION SLIDES

# Calculating basic summary statistics
volatility = returns.std()
mu = returns.mean()
cov_mat = returns.cov()

# Setting up return range
return_range = np.linspace(0.05, 0.22, 200) # Using ~0.05 from expected AAPL return to ~0.22 from expected GOOG return
ret_list = []
risks = []
props = []

# Running optimization for range
for ret in return_range:

    # Recreating model for each range
    m = Model("Portfolio Optimization")
    m.setParam('OutputFlag', 0)

    # Adding decision variables and defining risk
    vars = pd.Series(m.addVars(stocks, lb = 0), index = stocks)
    portfolio_risk = cov_mat.dot(vars).dot(vars)

    # Objective function
    m.setObjective(portfolio_risk, GRB.MINIMIZE)

    # Constraints
    m.addConstr(vars.sum() == 1, name = 'Budget')
    m.addConstr(mu.dot(vars) == ret, name = 'Return Sim')

    # Running model
    m.update()
    m.optimize()

    # Saving results once an optimal solution is found
    if m.status == GRB.OPTIMAL:
        weights = pd.Series([v.x for v in vars], index = stocks) # Extracting solved values
        risks.append(np.sqrt(m.objVal))
        ret_list.append(mu.dot(weights)) # Extracting portfolio return
        props.append(weights)

**GRAPH SOLUTION:**

In [ ]:
# TEMPLATE FROM OPTIMIZATION SLIDES

# Building graph
plt.plot(risks, ret_list, 'b-', linewidth=2)
plt.xlabel('Daily Risk (Std Dev %)')
plt.ylabel('Daily Return (%)')
plt.title('Efficient Frontier')
plt.grid(True)
plt.show()

In [ ]:
# Report-readying the efficient frontier graph

# Identify the minimum risk point to display on the curve
min_risk_idx = np.argmin(risks)

# Slice the data at the minimum risk point
eff_risks = risks[min_risk_idx:]
eff_returns = ret_list[min_risk_idx:]

# Plot
plt.figure(figsize=(10, 6), dpi=300) # higher DPI for crisp lines
plt.style.use('seaborn-v0_8-whitegrid')

plt.plot(eff_risks, eff_returns, color='#1f77b4', linewidth=2.5, label='Efficient Frontier')

plt.title('Efficient Frontier: Optimized Portfolio Performance', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Daily Risk (Standard Deviation %)', fontsize=12)
plt.ylabel('Expected Daily Return (%)', fontsize=12)


plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}%'))
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}%'))

plt.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.show()

**STRATEGY SOLUTION:**

In [ ]:
# Creating an allocation table using ret_list, risk, and props
frontier_df = pd.DataFrame(props, columns = stocks)
frontier_df['Expected Return'] = ret_list
frontier_df['Risk'] = risks
# Each row shows the allocation proportion and expected overall return for each risk level

# Providing three allocation options

# Minimum risk portfolio
min_risk_idx = frontier_df['Risk'].idxmin()
# Targeted return portfolio
target_idx_ret = (frontier_df['Expected Return'] - 0.18).abs().idxmin() # CAN CHANGE 0.18 TO ANY RETURN BETWEEN 0.05 AND 0.23
# Targeted risk portfolio
target_idx_risk = (frontier_df['Risk'] - 1.8).abs().idxmin() # CAN CHANGE 1.8 TO ANY RISK BETWEEN 1.7 AND 2.03
# Maximum return portfolio
max_ret_idx = frontier_df['Expected Return'].idxmax()

for label, idx in [('Min Risk', min_risk_idx),
                   ('Target Return (0.18%)', target_idx_ret),
                   ('Target Risk (1.8%)', target_idx_risk),
                   ('Max Return', max_ret_idx)]:
    print(f"\n{label}:")
    print(frontier_df.loc[idx].round(4))

# Template code for anyone wanting to pull allocation for a specific risk:
# print(frontier_df.iloc[(frontier_df['Risk'] - target_risk).abs().idxmin()]) # Change [target_risk] to anything between 1.7 and 2.03


**Minimum risk (1.71%) portfolio:** expected daily return of 0.1312%
- **NVDA:** 0.0000 $\rightarrow$ **0%**
- **GOOG:** 0.4302 $\rightarrow$ **43.02%**
- **AVGO:** 0.0000 $\rightarrow$ **0%**
- **AAPL:** 0.3478 $\rightarrow$ **34.78%**
- **META:** 0.1949 $\rightarrow$ **19.49%**

**Moderate risk (1.8%) portfolio:** expected daily return of 0.1798%
- **NVDA:** 0.0110 $\rightarrow$ **1.10%**
- **GOOG:** 0.7124 $\rightarrow$ **71.24%**
- **AVGO:** 0.0149 $\rightarrow$ **1.49%**
- **AAPL:** 0.1580 $\rightarrow$ **15.80%**
- **META:** 0.1038 $\rightarrow$ **10.38%**

**Maximum return (0.22%) portfolio:** risk of 1.9892%
- **NVDA:** 0.0318 $\rightarrow$ **3.18%**
- **GOOG:** 0.9164 $\rightarrow$ **91.64%**
- **AVGO:** 0.0509 $\rightarrow$ **5.09%**
- **AAPL:** 0.0000 $\rightarrow$ **0%**
- **META:** 0.0009 $\rightarrow$ **0.09%**

### Suggestions for Conclusion

- Provide high-level overview of the range of risk/return for these 5 stocks
- Compare the stocks that are best for minimizing risk (in terms of allocation %) to the stocks that are best for maximizing return
    - Could potentially add some practical knowledge of the corresponding companies and why their current performance might lead to these numbers